In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset

class BiPlanarXrayDataset(Dataset):
    def __init__(self, patient_dirs, transform=None):
        self.patient_dirs = patient_dirs
        self.transform = transform

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_path = self.patient_dirs[idx]
        front = Image.open(os.path.join(patient_path, 'front.png')).convert('L')
        side = Image.open(os.path.join(patient_path, 'side.png')).convert('L')

        if self.transform:
            front = self.transform(front)
            side = self.transform(side)

        return front, side



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import transforms

# Step 2a: List all patient folders
root_dir = 'data'
all_patients = [os.path.join(root_dir, d) for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

# Step 2b: Split into train and val
train_dirs, val_dirs = train_test_split(all_patients, test_size=0.2, random_state=42)

# Step 2c: Define your transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Step 2d: Create dataset instances
train_dataset = BiPlanarXrayDataset(train_dirs, transform=transform)
val_dataset = BiPlanarXrayDataset(val_dirs, transform=transform)

# Step 2e: Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
